In [ ]:
### Task 2 : is price missing at random? (target: 10 min)

# Question: Are listings without a price different from listings with one?
# Hypothesis: listings with no price weren’t bookable on the scrape date, so they should have much lower availability.
# Proxy: availability_365 (the number of days the listing is available in the next year) and number_of_reviews_ltm (reviews in the last 12 months, a proxy for activity).
# Expected output: one table with 2 rows (priced / not priced) and 4 columns: number of listings, median availability_365, 
# share of listings with availability_365 == 0, and median number_of_reviews_ltm.

import pandas as pd
df = pd.read_parquet("../data/clean/listings_clean.parquet") 

output = (
    df
    .assign(price_status=lambda d: d["price_eur"].isna().map({False: "priced", True: "not priced"}),
            zero_avail=lambda d: d["availability_365"] == 0)
    .groupby("price_status", as_index=False)    # Issue 1: how to display "priced"/"not priced" in the output instead of True/False? -> see .map above
    .agg(nb_listings = ("id", "size"),      # Issue 2: I know missings are not taken into account in python pandas, which explains the 0 for missings. what's the word to include in this line to include it? -> size
         median_365 = ("availability_365", "median"), 
         share_zero_avail = ("zero_avail", "mean"), # Issue 3: looks like it is not taken into account -> "availability_365"==0 was checking text against 0, not the value
         median_reviews_ltm = ("number_of_reviews_ltm", "median"))
    .round({"share_zero_avail":3}) # Issue 4: round() applies to all columns here, but I didn't find where to place it within share_listings. -> add entries within round()
)
display(output) 

# Sanity check (as code): the two counts add up to len(df), and the “True” count equals 33,612. 
assert output["nb_listings"].sum()==len(df)
assert output.loc[ output["price_status"]=="not priced" , "nb_listings"].item()==33612 # Issue: the output.loc returns a part of the table so not an integer. How to convert it? -> add .item() to extract only the cell's value

# Then, in up to 2 sentences: what do you conclude for the analysis, and should you drop, fill or keep the missing prices? 
# Missing prices are not random: 76.4% of unpriced listings have zero availability over the next year (vs 1.5% of priced ones), 
# and their median number of reviews in the last 12 months is 0 against 4, so they are mostly dormant listings. 
# I’d keep them in the data but never fill their price, and state that every price statistic describes active listings only.

## Rules
# When to use a lambda
    # df["col"] refers to the original table, before any step of the chain.
    # lambda d: d["col"] refers to the table as it is at that step.
    # The simple habit: inside a chain, always use lambda d:. It’s never wrong, while df[...] is right only by luck.


,price_status,nb_listings,median_365,share_zero_avail,median_reviews_ltm
0,not priced,33612,0.0,0.764,0.0
1,priced,43537,238.0,0.015,4.0
